In [ ]:
import numpy as np
import pandas as pd

# Modern sklearn: set global output to pandas for dataframe-friendly transforms
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.head()

# Let's Plan

In [ ]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

In [ ]:
# Step 1 -> train/test/split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Survived']),
    df['Survived'],
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train.head()

In [ ]:
y_train.sample(5)

In [ ]:
# Imputation transformer — using column names (modern best practice)
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), ['Age']),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), ['Embarked'])
], remainder='passthrough', verbose_feature_names_out=False)

In [ ]:
# One hot encoding — sparse_output=False replaces deprecated sparse=False
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['Sex', 'Embarked'])
], remainder='passthrough', verbose_feature_names_out=False)

In [ ]:
# Scaling
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 10))
])

In [ ]:
# Feature selection
trf4 = SelectKBest(score_func=chi2, k=8)

In [ ]:
# Train the model
trf5 = DecisionTreeClassifier()

# Create Pipeline

In [ ]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5)
])

# Pipeline Vs make_pipeline

Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

In [ ]:
# Alternate Syntax
pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)

In [ ]:
# Train
pipe.fit(X_train, y_train)

# Explore the Pipeline

In [ ]:
# Named steps (works with Pipeline, not make_pipeline)
pipe.named_steps

In [ ]:
# Display Pipeline as interactive diagram
from sklearn import set_config
set_config(display='diagram')
pipe

In [ ]:
# Predict
y_pred = pipe.predict(X_test)

In [ ]:
y_pred

In [ ]:
accuracy_score(y_test, y_pred)

# Cross Validation using Pipeline

In [ ]:
# Cross validation using cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

# GridSearch using Pipeline

In [ ]:
# GridSearchCV — access pipeline step params via double underscore
params = {
    'decisiontreeclassifier__max_depth': [1, 2, 3, 4, 5, None]
}

In [ ]:
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

In [ ]:
grid.best_score_

In [ ]:
grid.best_params_

# Exporting the Pipeline

In [ ]:
# Export using joblib (recommended over pickle for sklearn objects)
import joblib
joblib.dump(pipe, 'pipe.pkl')